## Upgrade Scikit-Learn

In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0'

## Load the Digit Recognizer dataset

In [ ]:
import pathlib

import pandas as pd


DATA_DIR = pathlib.Path("/kaggle/input/digit-recognizer")


train_df = pd.read_csv(DATA_DIR / "train.csv")
train_features_df = train_df.drop("label", axis=1)
train_labels = train_df.loc[:, "label"]

## SGDClassifier Benchmark

In [ ]:
import numpy as np
from sklearn import decomposition, linear_model, preprocessing, pipeline


RANDOM_STATE = np.random.RandomState(42)


sgd_classifier = pipeline.make_pipeline(
    preprocessing.StandardScaler(
        with_mean=False,
        with_std=True
    ),
    decomposition.PCA(
        n_components=0.95,
        random_state=RANDOM_STATE,
        whiten=True
    ),
    linear_model.SGDClassifier(
        early_stopping=True,
        loss="log_loss",
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
)

In [ ]:
%%timeit -n 1 -r 1

_ = sgd_classifier.fit(
    train_features_df,
    train_labels
)

In [ ]:
from sklearn import metrics


train_predictions = sgd_classifier.predict(train_features_df)

train_accuracy = metrics.accuracy_score(
    train_labels,
    train_predictions
)

print(f"Training accuracy: {train_accuracy}")

In [ ]:
from sklearn import model_selection


cv_accuracy = model_selection.cross_val_score(
    sgd_classifier,
    train_features_df,
    train_labels,
    cv=4,
    n_jobs=-1,
    scoring="accuracy",
    verbose=1
)

print(f"CV accuracy: {cv_accuracy.mean()}")

## Flagging Outliers using IsolationForest

In [ ]:
from sklearn import base, ensemble


class IsolationForestOutlierFlagger(base.BaseEstimator, base.TransformerMixin):
    
    def __init__(self, contamination="auto", random_state=None, dtype=np.float64):
        self.contamination = contamination
        self.random_state = random_state
        self.dtype = dtype

    def fit(self, X, y=None):
        self.detector_ = ensemble.IsolationForest(
            contamination=self.contamination,
            random_state=self.random_state,
            n_jobs=-1
        )
        self.detector_.fit(X)
        return self

    def transform(self, X, y=None):
        # predict returns 1 for inliers, -1 for outliers
        predicted_outliers = (
            self.detector_
                .predict(X)
        )

        # create an outlier flag
        is_outlier = predicted_outliers == -1
        is_outlier = (
            is_outlier.astype(self.dtype)
                      .reshape(-1, 1)
        )    
                
        # append flag as an extra feature
        X_out = np.hstack([X, is_outlier])
        
        return X_out

In [ ]:
outlier_flagger = pipeline.make_pipeline(
    preprocessing.StandardScaler(),
    IsolationForestOutlierFlagger(
        random_state=RANDOM_STATE,
    ),
)


In [ ]:
prepared_train_features = outlier_flagger.fit_transform(train_features_df)

In [ ]:
prepared_train_features.shape

In [ ]:
number_outliers = np.sum(prepared_train_features[:, -1])
print(number_outliers)

In [ ]:
import matplotlib.pyplot as plt


is_outlier = prepared_train_features[:, -1] == 1
outlier_train_features = train_features_df.iloc[is_outlier, :]
outlier_train_labels = train_labels.iloc[is_outlier]

fig, axes = plt.subplots(1, 10, figsize=(15, 5), sharex=True, sharey=True)
for i in range(10):
    k = RANDOM_STATE.randint(len(outlier_train_features))
    axes[i].imshow(outlier_train_features.iloc[k, :].to_numpy().reshape(28, 28))
    axes[i].set_title(outlier_train_labels.iloc[k])

In [ ]:
sgd_classifier = pipeline.make_pipeline(
    preprocessing.StandardScaler(
        with_mean=False,
        with_std=True,
    ),
    IsolationForestOutlierFlagger(
        random_state=RANDOM_STATE,
    ),
    decomposition.PCA(
        n_components=0.95,
        random_state=RANDOM_STATE,
        whiten=True,
    ),
    linear_model.SGDClassifier(
        early_stopping=True,
        loss="log_loss",
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
)


In [ ]:
_ = sgd_classifier.fit(
    train_features_df,
    train_labels
)

In [ ]:
train_predictions = sgd_classifier.predict(train_features_df)

train_accuracy = metrics.accuracy_score(
    train_labels,
    train_predictions
)

print(f"Training accuracy: {train_accuracy}")

In [ ]:
cv_accuracy = model_selection.cross_val_score(
    sgd_classifier,
    train_features_df,
    train_labels,
    cv=4,
    n_jobs=-1,
    scoring="accuracy",
    verbose=1
)

print(f"CV accuracy: {cv_accuracy.mean()}")

## Flagging Outliers using LocalOutlierFactor

In [ ]:
from sklearn import neighbors


class LocalOutlierFactorOutlierFlagger(base.BaseEstimator, base.TransformerMixin):
    
    def __init__(self, n_neighbors=20, contamination="auto", dtype=np.float64):
        self.n_neighbors = n_neighbors
        self.contamination = contamination
        self.dtype = dtype

    def fit(self, X, y=None):
        self.detector_ = neighbors.LocalOutlierFactor(
            n_neighbors=self.n_neighbors,
            contamination=self.contamination,
            novelty=True,
            n_jobs=-1,
        )
        self.detector_.fit(X)
        return self

    def transform(self, X, y=None):
        # predict returns 1 for inliers, -1 for outliers
        predicted_outliers = (
            self.detector_
                .predict(X)
        )

        # create an outlier flag
        is_outlier = predicted_outliers == -1
        is_outlier = (
            is_outlier.astype(self.dtype)
                      .reshape(-1, 1)
        )    
                
        # append flag as an extra feature
        X_out = np.hstack([X, is_outlier])
        
        return X_out


In [ ]:
outlier_flagger = pipeline.make_pipeline(
    preprocessing.StandardScaler(
        with_mean=False,
        with_std=True,
    ),
    LocalOutlierFactorOutlierFlagger(),
)


In [ ]:
prepared_train_features = outlier_flagger.fit_transform(train_features_df)

In [ ]:
is_outlier = prepared_train_features[:, -1] == 1
outlier_train_features = train_features_df.iloc[is_outlier]
outlier_train_labels = train_labels.iloc[is_outlier]

fig, axes = plt.subplots(1, 10, figsize=(15, 5), sharex=True, sharey=True)
for i in range(10):
    k = RANDOM_STATE.randint(len(outlier_train_features))
    axes[i].imshow(outlier_train_features.iloc[k, :].to_numpy().reshape(28, 28))
    axes[i].set_title(outlier_train_labels.iloc[k])

In [ ]:
features_preparation_pipeline = pipeline.make_pipeline(
    preprocessing.StandardScaler(
        with_mean=False,
        with_std=True,
    ),
    LocalOutlierFactorOutlierFlagger(),
        decomposition.PCA(
        n_components=0.95,
        random_state=RANDOM_STATE,
        whiten=True,
    ),
)


In [ ]:
%%timeit -n 1 -r 1


prepared_train_features = features_preparation_pipeline.fit_transform(
    train_features_df
)

In [ ]:
sgd_classifier = linear_model.SGDClassifier(
    early_stopping=True,
    loss="log_loss",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

In [ ]:
%%timeit -n 1 -r 1


_ = sgd_classifier.fit(
    prepared_train_features,
    train_labels
)

In [ ]:
train_predictions = sgd_classifier.predict(prepared_train_features)

train_accuracy = metrics.accuracy_score(
    train_labels,
    train_predictions
)

print(f"Training accuracy: {train_accuracy}")

In [ ]:
cv_accuracy = model_selection.cross_val_score(
    sgd_classifier,
    prepared_train_features,
    train_labels,
    cv=4,
    n_jobs=-1,
    scoring="accuracy",
    verbose=1
)

print(f"CV accuracy: {cv_accuracy.mean()}")